In [10]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

def load_and_clean(filename):
    return np.loadtxt(filename, skiprows=2)

data_x = load_and_clean('x24x24.txt')
data_y = load_and_clean('y24x24.txt')
data_z = load_and_clean('z24x24.txt')
full_data = np.vstack([data_x, data_y, data_z])

X = full_data[:, :576]
y = full_data[:, 578]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=67,
    stratify=y           # photos of a person will be in both sets
)

In [11]:
from scipy.ndimage import rotate, shift
import numpy as np
import pandas as pd

def augment_image_flat(x):
    img = x.reshape(24, 24)

    # mały obrót
    angle = np.random.uniform(-5, 5)
    img = rotate(img, angle, reshape=False, mode="nearest")

    # małe przesunięcie max 1-2 piksele
    dx = np.random.uniform(-1, 1)
    dy = np.random.uniform(-1, 1)
    img = shift(img, shift=(dy, dx), mode="nearest")

    return img.reshape(-1)


def augument_df(X_train, y_train):
    X_aug = [X_train]
    y_aug = [y_train]


    unique_classes, counts = np.unique(y_train, return_counts=True)
    target = 100
    for y, count in zip(unique_classes,counts):
       X = X_train[y_train==y]
       to_fill = target-count
       new_img = []
       for c in range(to_fill):
            img = X[np.random.randint(len(X))]
            new_img.append(augment_image_flat(img))
       if len(new_img)>0:
            X_aug.append(np.array(new_img))
            y_aug.append(np.full(to_fill, y))

    return np.vstack(X_aug), np.concatenate(y_aug)


In [12]:
batch_size = 64
learning_rate = 0.001
epochs = 100

X_train, y_train = augument_df(X_train, y_train)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [13]:
class ANN(nn.Module):
    def __init__(self, input_size=576, hidden_size=128, num_classes=48):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # nn.Linear(200, hidden_size),
            # nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

In [14]:
num_classes = len(np.unique(y))

model = ANN(
    input_size=576,
    hidden_size=128,
    num_classes=num_classes
)

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [16]:
train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

for epoch in range(epochs):
    #training
    model.train()

    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * X_batch.size(0)

        predictions = torch.argmax(outputs, dim=1)
        correct_train += (predictions == y_batch).sum().item()
        total_train += y_batch.size(0)

    epoch_train_loss = running_train_loss / total_train
    epoch_train_accuracy = correct_train / total_train

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)

   # validation
    model.eval()

    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_val_loss += loss.item() * X_batch.size(0)

            predictions = torch.argmax(outputs, dim=1)
            correct_val += (predictions == y_batch).sum().item()
            total_val += y_batch.size(0)

    epoch_val_loss = running_val_loss / total_val
    epoch_val_accuracy = correct_val / total_val

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_accuracy)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train loss: {epoch_train_loss:.4f} | "
            f"val loss: {epoch_val_loss:.4f} | "
            f"train acc: {epoch_train_accuracy:.4f} | "
            f"val acc: {epoch_val_accuracy:.4f}"
        )

Epoch 10/100 | train loss: 2.0760 | val loss: 1.7420 | train acc: 0.3798 | val acc: 0.5135
Epoch 20/100 | train loss: 1.6681 | val loss: 1.3676 | train acc: 0.4923 | val acc: 0.6079
Epoch 30/100 | train loss: 1.4295 | val loss: 1.2521 | train acc: 0.5675 | val acc: 0.6445
Epoch 40/100 | train loss: 1.3014 | val loss: 1.1378 | train acc: 0.6054 | val acc: 0.6650
Epoch 50/100 | train loss: 1.1697 | val loss: 1.0682 | train acc: 0.6351 | val acc: 0.6971
Epoch 60/100 | train loss: 1.1038 | val loss: 1.0349 | train acc: 0.6520 | val acc: 0.7045
Epoch 70/100 | train loss: 1.0625 | val loss: 1.0274 | train acc: 0.6602 | val acc: 0.7074
Epoch 80/100 | train loss: 1.0092 | val loss: 1.0202 | train acc: 0.6857 | val acc: 0.7220
Epoch 90/100 | train loss: 0.9321 | val loss: 1.0401 | train acc: 0.7058 | val acc: 0.7125
Epoch 100/100 | train loss: 0.9195 | val loss: 1.0058 | train acc: 0.7099 | val acc: 0.7228


In [17]:
# for epoch in range(epochs):
#     model.train()
#     running_loss = 0.0
#
#     for X_batch, y_batch in train_loader:
#         optimizer.zero_grad()
#
#         outputs = model(X_batch)
#         loss = criterion(outputs, y_batch)
#
#         loss.backward()
#         optimizer.step()
#
#         running_loss += loss.item()
#
#     avg_loss = running_loss / len(train_loader)
#
#     if (epoch + 1) % 10 == 0:
#         print(f"Epoch {epoch + 1}/{epochs}, loss: {avg_loss:.4f}")

In [18]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        outputs = model(X_batch)
        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct / total

print(f"Validation accuracy: {accuracy * 100:.2f}%")

Validation accuracy: 72.28%
